<a href="https://colab.research.google.com/github/hailehong2002/DQN-Agent-ALE-Freeway-v5/blob/main/notebooks/sessions/7_dqn/dqn_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo](https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/shared_assets/logo.png)

# Practice 7 Homework: Train a DQN Agent on a Non-Pong Atari Environment

**Developers:** Zoltan Barta  
**Date:** 2026-04-13  
**Version:** 2025-26/2

[<img src="https://colab.research.google.com/assets/colab-badge.svg">](https://colab.research.google.com/github/BartaZoltan/deep-reinforcement-learning-course/blob/main/notebooks/sessions/7_dqn/dqn_homework.ipynb)

## Task Description

In this homework, your task is to choose **one Atari environment other than Pong** and train a **Deep Q-Network (DQN)** agent on it.

You should implement the full pipeline yourself. This includes environment creation and preprocessing, the replay buffer, the Q-network, the DQN target computation, the training loop, and the final evaluation. No starter implementation is provided on purpose: the goal is to transfer the ideas from the session notebook into a new Atari setting independently.

Use the same overall DQN logic as in the practice material: image-based Atari observations, standard DQN preprocessing, experience replay, a target network, epsilon-greedy exploration, and greedy evaluation.

`Pong` is not allowed for this homework.


In [1]:
# Help cell: list Atari environments that are allowed for this homework.
try:
    import gymnasium as gym
    import ale_py
except Exception as exc:
    raise ImportError(
        "This homework requires gymnasium with Atari support and ale-py."
    ) from exc

gym.register_envs(ale_py)

atari_env_ids = sorted(
    spec.id
    for spec in gym.registry.values()
    if spec.id.startswith("ALE/")
    and spec.id.endswith("-v5")
    and "Pong" not in spec.id
)

print(f"Available Atari environments excluding Pong: {len(atari_env_ids)}")
for env_id in atari_env_ids:
    print(" -", env_id)

recommended = [
    env_id
    for env_id in [
        "ALE/Breakout-v5",
        "ALE/Boxing-v5",
        "ALE/Freeway-v5",
        "ALE/Enduro-v5",
    ]
    if env_id in atari_env_ids
]

print("\nRecommended starter choices:")
for env_id in recommended:
    print(" -", env_id)


Available Atari environments excluding Pong: 103
 - ALE/Adventure-v5
 - ALE/AirRaid-v5
 - ALE/Alien-v5
 - ALE/Amidar-v5
 - ALE/Assault-v5
 - ALE/Asterix-v5
 - ALE/Asteroids-v5
 - ALE/Atlantis-v5
 - ALE/Atlantis2-v5
 - ALE/Backgammon-v5
 - ALE/BankHeist-v5
 - ALE/BasicMath-v5
 - ALE/BattleZone-v5
 - ALE/BeamRider-v5
 - ALE/Berzerk-v5
 - ALE/Blackjack-v5
 - ALE/Bowling-v5
 - ALE/Boxing-v5
 - ALE/Breakout-v5
 - ALE/Carnival-v5
 - ALE/Casino-v5
 - ALE/Centipede-v5
 - ALE/ChopperCommand-v5
 - ALE/CrazyClimber-v5
 - ALE/Crossbow-v5
 - ALE/Darkchambers-v5
 - ALE/Defender-v5
 - ALE/DemonAttack-v5
 - ALE/DonkeyKong-v5
 - ALE/DoubleDunk-v5
 - ALE/Earthworld-v5
 - ALE/ElevatorAction-v5
 - ALE/Enduro-v5
 - ALE/Entombed-v5
 - ALE/Et-v5
 - ALE/FishingDerby-v5
 - ALE/FlagCapture-v5
 - ALE/Freeway-v5
 - ALE/Frogger-v5
 - ALE/Frostbite-v5
 - ALE/Galaxian-v5
 - ALE/Gopher-v5
 - ALE/Gravitar-v5
 - ALE/Hangman-v5
 - ALE/HauntedHouse-v5
 - ALE/Hero-v5
 - ALE/HumanCannonball-v5
 - ALE/IceHockey-v5
 - ALE/Ja

**IMPORT AND DEVICE**

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import matplotlib.pyplot as plt
import imageio

gym.register_envs(ale_py)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [2]:
def _select_eval_action(agent, obs):
    if hasattr(agent, "predict"):
        return int(agent.predict(obs))
    if hasattr(agent, "act"):
        return int(agent.act(obs))
    if callable(agent):
        return int(agent(obs))
    raise TypeError(
        "Unsupported agent interface. Provide either a callable agent(obs), "
        "an object with agent.act(obs), or an object with agent.predict(obs)."
    )


def evaluate_agent(agent, make_env_fn, n_episodes=10, seed=42, max_steps=5_000):
    """Evaluate a trained agent on several independent episodes.

    Parameters
    ----------
    agent : object or callable
        The trained policy to evaluate.

        This helper accepts any of the following interfaces:
        - `agent(obs) -> action`
        - `agent.act(obs) -> action`
        - `agent.predict(obs) -> action`

        The returned action must be convertible to `int` and must be valid for
        the environment.


    make_env_fn : callable
        A factory that creates a fresh environment for evaluation.

        Recommended pattern:
        ```python
        def make_env_fn(seed=None):
            return make_atari_env("ALE/Breakout-v5", seed=seed)
        ```

        The helper will try to call `make_env_fn(seed=...)`. If your factory
        does not accept a `seed` argument, it will fall back to calling
        `make_env_fn()`.

        The environment returned by `make_env_fn` should already include the
        same preprocessing and wrappers that you used during training.

    n_episodes : int, default=10
        Number of independent evaluation episodes.

    seed : int, default=42
        Base random seed. Episode `i` uses `seed + i`.

    max_steps : int, default=5000
        Maximum number of interaction steps per episode. This prevents the
        evaluation from running forever if an episode is unusually long.

    Returns
    -------
    dict
        A dictionary with the raw returns and lengths, plus aggregate summary
        statistics:
        - `returns`
        - `lengths`
        - `mean_return`
        - `std_return`
        - `mean_length`
        - `std_length`

    How to use
    ----------
    1. Finish training your DQN agent.
    2. Write a small environment factory that builds one fresh evaluation env.
    3. Pass either your greedy agent directly, or wrap it into a callable.
    4. Call `evaluate_agent(...)` and report the returned mean and standard
       deviation values in your notebook.

    Minimal example:
    ```python
    def make_env_fn(seed=None):
        return make_atari_env("ALE/Breakout-v5", seed=seed)

    def greedy_agent(obs):
        return select_action(q_network, obs, epsilon=0.0)

    eval_stats = evaluate_agent(greedy_agent, make_env_fn, n_episodes=10)
    print(eval_stats)
    ```
    """
    import numpy as np

    returns = []
    lengths = []

    for episode_idx in range(n_episodes):
        episode_seed = seed + episode_idx
        try:
            env = make_env_fn(seed=episode_seed)
        except TypeError:
            env = make_env_fn()

        obs, _ = env.reset(seed=episode_seed)
        episode_return = 0.0
        episode_length = 0

        for _ in range(max_steps):
            action = _select_eval_action(agent, obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_return += float(reward)
            episode_length += 1

            if terminated or truncated:
                break

        returns.append(episode_return)
        lengths.append(episode_length)
        env.close()

    returns = np.asarray(returns, dtype=float)
    lengths = np.asarray(lengths, dtype=float)

    return {
        "returns": returns,
        "lengths": lengths,
        "mean_return": float(np.mean(returns)),
        "std_return": float(np.std(returns)),
        "mean_length": float(np.mean(lengths)),
        "std_length": float(np.std(lengths)),
    }


## Evaluation Requirement

For submission, your notebook should contain the following for **one Atari environment other than Pong**:

- the exact environment ID you used,
- one completed DQN training run on that environment,
- greedy evaluation on **at least 10 independent episodes**,
- the final `mean return ± std return`,
- the final `mean episode length ± std episode length`,
- at least one training curve and one greedy rollout GIF,
- a short discussion of whether the learned behavior is clearly better than random play.

`Pong` submissions do not satisfy this homework requirement.
